## 0 · Vertrekpunt: waar de vorige les eindigde.

Tot nu toe was de vraag aan het systeem een titel. Vandaag wordt het een persoon. Deel 1 en 2 laten het principe zien: een profiel is het gemiddelde van een aantal rijen uit `X`. Daarna bouw je een profiel uit wat een lezer invult: trefwoorden, genres en eigen woorden. Eén functie `recommend_for` maakt van de hele vragenlijst een lijst, en een koude start vangt de lezer op die niets invulde.

De dummydata heeft expres andere kolomnamen dan jullie catalogus (`title`, `summary`, `tags`, `category`): vertalen, niet kopiëren.

| Deel | Demo | Opdracht |
|---|---|---|
| 0 | Vertrekpunt: `build_vectors`, `X`, `similar_items` | – |
| 1 | Een profiel van één titel; `recommend` | – |
| 2 | Meer titels → één gemiddelde; `build_profile` | – |
| 3 | Drie bronnen: trefwoorden, genres, eigen woorden | Opdracht 1: `profile_from_tags` en `profile_from_text` |
| 4 | De vragenlijst als dict | Opdracht 2: `recommend_for(person)` voor elke combinatie van velden |
| 5 | Koude start | Opdracht 3: kies een aanpak en bouw `cold_start` |

Dit is de code die in jullie `recommender.py` ook staat, in één cel zodat dit notebook zelfstandig draait. Twee verschillen met de vorige les:

- `build_vectors` heeft nu drie blokken: omschrijving, genre én trefwoorden.
- `build_vectors` geeft ook de vectorizers terug. Die heb je nodig om een losse tekst door dezelfde TF-IDF te halen.

In [9]:
import re
import numpy as np
import pandas as pd

books = pd.DataFrame({
    "title": [
        "De draak onder de school", "Leven in de ruimte", "Het drakenei", "Nieuw in de stad",
        "Code rood op de redactie", "Vulkanen van dichtbij", "De laatste wolf",
        "Mijn broer de robot", "Een zomer op Terschelling", "Bouw je eigen game",
        "De verdwenen schat van kapitein Bos", "Alles over bijen", "Het geheim van de oude molen",
        "Eerste hulp bij heimwee", "Draken bestaan niet", "Op zoek naar Mars", "De voetbalschool",
        "Koken zonder recept", "Het meisje dat niet kon stoppen met rennen", "Waarom de zee zout is",
        "Sterren kijken voor beginners", "De ruimtereis van Juul", "Slimme machines",
        "Dieren in de winter", "De draak van de brandweer", "Vriendschap op kamp",
        "Het raadsel van de vuurtoren", "Bakken met kinderen", "Wolven in Europa",
        "De schoolreis naar de bergen",
    ],
    "summary": [
        "Een meisje ontdekt dat de oude draak onder haar school niet dood is. Alleen zij kan hem horen, en elke nacht wordt hij onrustiger. Samen met haar beste vriend zoekt ze uit hoe ze de draak kan tegenhouden voordat de school instort. Maar niet iedereen op school wil dat de draak verdwijnt.",
        "Hoe eten, slapen en douchen astronauten aan boord van het ruimtestation? Dit boek neemt je mee achter de schermen, van de training op aarde tot de terugkeer na een half jaar in de ruimte. Met veel foto's en antwoorden op vragen van kinderen.",
        "Twee broers vinden in het bos een enorm ei dat warm aanvoelt. Als het uitkomt, staat het hele dorp op zijn kop. De jongens moeten het geheim bewaren, want de burgemeester wil alles wat vreemd is laten opruimen. Een spannend verhaal over broers, een geheim en een verrassing die groter is dan ze dachten.",
        "Na de scheiding van haar ouders verhuist Noor naar de stad. Op een nieuwe school kent ze niemand en alles is anders dan thuis. Langzaam leert ze nieuwe vrienden kennen, maar ze mist haar oude dorp en haar vader. Een verhaal over verhuizen, vriendschap en opnieuw beginnen.",
        "Op de schoolkrant lekt een geheim uit. Wie heeft gepraat, en hoe houd je de redactie bij elkaar? Quinten is hoofdredacteur en moet kiezen tussen de waarheid en zijn vrienden. Een verhaal over school, vriendschap en de macht van een krant.",
        "Waarom barsten vulkanen uit en hoe meten onderzoekers dat? In dit boek lees je wat er onder de grond gebeurt en wat je doet als er een vulkaan in de buurt actief wordt. Met verhalen van vulkanologen die dicht bij de lava werken.",
        "In een dorp in de bergen wordt de laatste wolf gezien. Een jongen wil de wolf redden, maar de boeren willen hem weg. Terwijl de winter dichterbij komt, moet de jongen kiezen tussen zijn dorp en het dier. Een verhaal over vriendschap tussen mens en dier.",
        "Sems ouders kopen een robot die alles beter kan dan hij. Op school, thuis en op het voetbalveld lijkt de robot overal beter in. Totdat de robot iets doet wat niemand had verwacht. Een grappig verhaal over broers, techniek en wat een mens bijzonder maakt.",
        "Drie vrienden werken een zomer lang in een strandtent op Terschelling. Ze ontdekken dat het eiland een geheim bewaart, verstopt tussen de duinen. Wat begint als een vakantiebaantje, wordt een zoektocht naar een oude schat.",
        "Stap voor stap een eigen computerspel maken, van idee tot speelbare versie. Je leert programmeren zonder ervaring, ontwerpt je eigen personages en test je spel met vrienden. In dit boek maak je elke week een nieuw level.",
        "Een oude kaart, een verdwenen schip en een klas die op schoolreis per ongeluk een echte schat vindt. Maar de schat is niet van hen, en iemand anders zoekt er al jaren naar. Een spannend verhaal vol raadsels, geheime gangen en piraten.",
        "Hoe werkt een bijenvolk en waarom verdwijnen bijen? In dit boek lees je wat een koningin doet, hoe bijen honing maken en wat je in je eigen tuin kunt doen om ze te helpen. Met proefjes en weetjes.",
        "De oude molen aan de rand van het dorp draait soms 's nachts. Lotte en haar buurjongen gaan op onderzoek uit en vinden een verborgen kamer. Wie woonde hier vroeger, en waarom mocht niemand het weten? Een spannend verhaal over een geheim dat al honderd jaar bewaard wordt.",
        "Tips voor als je voor het eerst van huis bent, op kamp, op schoolreis of in een gastgezin. Wat doe je als je je verdrietig voelt en wie kun je dan om hulp vragen? Een boek vol praktische tips en verhalen van andere kinderen.",
        "Een jongen die niet in draken gelooft, krijgt er een in zijn tuin. En de draak gelooft niet in jongens. Samen komen ze erachter dat ze meer op elkaar lijken dan ze dachten. Een grappig verhaal over vriendschap en over geloven in dingen die je niet kunt zien.",
        "Wat weten we over de rode planeet? In dit boek lees je hoe de missies naar Mars verlopen, wat robots daar ontdekken en of er ooit mensen zullen wonen. Met foto's uit de ruimte en tekeningen van toekomstige steden.",
        "Op de voetbalschool moet Amir kiezen tussen zijn beste vriend en een plek in het eerste team. Op het veld wordt de strijd steeds harder en op school raakt hij zijn vrienden kwijt. Een verhaal over sport, vriendschap en eerlijk spelen.",
        "Koken op gevoel: wat past bij wat, hoe je proeft en hoe je bijstuurt. Met twintig basisgerechten die je steeds anders kunt maken. In dit boek leer je koken zonder dat je alles hoeft af te wegen.",
        "Fenna rent elke dag harder en verder, tot haar lichaam zegt dat het genoeg is geweest. Haar trainer ziet het niet en haar vriendin durft niets te zeggen. Een eerlijk verhaal over sport, druk en het durven stoppen.",
        "Korte antwoorden op grote vragen over de zee: waarom is het water zout, hoe ontstaan getijden en wat leeft er op de bodem? Met proefjes die je thuis kunt doen en weetjes over stromingen.",
        "Hoe vind je de Grote Beer, en wat zie je door een telescoop? In dit boek leer je de sterren en planeten herkennen, ook als je in de stad woont. Met kaarten van de sterrenhemel en tips om zelf de ruimte te bekijken.",
        "Juul wil astronaut worden en mag meedoen aan een training voor kinderen. Maar tijdens de proef gaat er iets mis in het ruimtestation. Een spannend verhaal over durf, samenwerken en dromen van een reis naar de sterren.",
        "Wat kan een robot en wat kan hij nog niet? In dit boek lees je hoe computers leren, hoe robots worden gebouwd en wat dat betekent voor de toekomst. Met opdrachten om zelf een eenvoudige robot te programmeren.",
        "Wat doet een egel als het vriest, en waar gaan vogels naartoe? In dit boek lees je hoe dieren de winter doorkomen, van de winterslaap tot de lange reis naar het zuiden. Met foto's, weetjes en tips om dieren te helpen in je eigen tuin.",
        "In het stadje woont een jonge draak die per ongeluk brandweerman wordt. Hij blust branden met zijn adem, maar kan zelf geen vuur meer maken. Een grappig verhaal over een draak die zijn plek zoekt.",
        "Tijdens het schoolkamp raakt Mila haar beste vriendin kwijt aan de nieuwe leerling. Ze voelt zich buitengesloten en heeft heimwee naar huis. Een verhaal over kamp, vriendschap en jaloezie.",
        "Op een eiland staat een vuurtoren waar al jaren niemand meer woont. Maar elke nacht brandt er licht. Twee vrienden gaan op onderzoek uit en ontdekken een geheim dat het eiland bijna verloor.",
        "Van pannenkoeken tot taart: twintig recepten om zelf te maken, met uitleg stap voor stap. In dit boek leer je hoe je deeg maakt, hoe je proeft en wat je doet als iets mislukt.",
        "Waar leven wolven en waarom keren ze terug naar Nederland? In dit boek lees je hoe wolven leven, hoe ze jagen en hoe mensen en wolven kunnen samenleven. Met verhalen van boswachters en boeren.",
        "Een klas gaat op schoolreis naar de bergen, maar een sneeuwstorm verandert alles. Samen met de juf moet Daan met zijn klasgenoten een nacht overleven in een oude hut. Een spannend verhaal over samenwerken en over wat vriendschap waard is.",
    ],
    "tags": [
        "draak;school;vriendschap", "ruimte;astronaut;ruimtestation", "draak;bos;broers;geheim",
        "verhuizen;vriendschap;scheiding", "school;krant;vriendschap", "vulkaan;lava;onderzoek",
        "wolf;bergen;dieren", "robot;techniek;broers", "eiland;zomer;schat",
        "game;programmeren;computer", "schat;piraten;schoolreis", "bijen;natuur;tuin",
        "molen;geheim;dorp", "heimwee;kamp;tips", "draak;vriendschap;humor", "mars;ruimte;robots",
        "voetbal;sport;vriendschap", "koken;recepten", "hardlopen;sport;druk", "zee;water;proefjes",
        "sterren;telescoop;ruimte", "ruimte;astronaut;training", "robot;computer;toekomst",
        "dieren;winter;natuur", "draak;brandweer;humor", "kamp;vriendschap;jaloezie",
        "vuurtoren;eiland;geheim", "bakken;recepten;koken", "wolf;natuur;nederland",
        "schoolreis;bergen;sneeuw",
    ],
    "category": [
        "fantasy", "informatief", "", "realistisch", "realistisch", "informatief", "realistisch",
        "humor", "", "informatief", "avontuur", "informatief", "spanning", "informatief", "humor",
        "informatief", "realistisch", "informatief", "", "informatief", "informatief", "avontuur",
        "informatief", "informatief", "humor", "", "spanning", "informatief", "informatief", "",
    ],
})
books["category_list"] = books["category"].apply(lambda c: [c] if c else [])
books["tag_list"] = books["tags"].str.split(";")
titles = books["title"].tolist()
books.head()

,title,summary,tags,category,category_list,tag_list
0,De draak onder de school,Een meisje ontdekt dat de oude draak onder haa...,draak;school;vriendschap,fantasy,[fantasy],"[draak, school, vriendschap]"
1,Leven in de ruimte,"Hoe eten, slapen en douchen astronauten aan bo...",ruimte;astronaut;ruimtestation,informatief,[informatief],"[ruimte, astronaut, ruimtestation]"
2,Het drakenei,Twee broers vinden in het bos een enorm ei dat...,draak;bos;broers;geheim,,[],"[draak, bos, broers, geheim]"
3,Nieuw in de stad,Na de scheiding van haar ouders verhuist Noor ...,verhuizen;vriendschap;scheiding,realistisch,[realistisch],"[verhuizen, vriendschap, scheiding]"
4,Code rood op de redactie,Op de schoolkrant lekt een geheim uit. Wie hee...,school;krant;vriendschap,realistisch,[realistisch],"[school, krant, vriendschap]"


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)          # eenmalig, daarna uit de lokale cache

stop_words = set(stopwords.words("dutch"))
EIGEN_STOPWOORDEN = {"boek", "verhaal", "jongen", "meisje", "jaar"}
STOPWORDS = stop_words | EIGEN_STOPWOORDEN

def tokenize_nostop(text):
    return [t for t in re.findall(r"\w+", str(text).lower()) if t not in STOPWORDS]

def split_semicolon(s):
    return [p.strip() for p in s.split(";") if p.strip()]

def build_vectors(df, weights):
    """Return (X, vectorizers): one sparse row per title, and the fitted vectorizers per block."""
    vec = {
        "summary":  TfidfVectorizer(tokenizer=tokenize_nostop, token_pattern=None, lowercase=False),
        "category": CountVectorizer(tokenizer=split_semicolon, token_pattern=None, lowercase=True, binary=True),
        "tags":     CountVectorizer(tokenizer=split_semicolon, token_pattern=None, lowercase=True, binary=True),
    }
    blocks = []
    for col, v in vec.items():
        M = v.fit_transform(df[col].fillna(""))
        blocks.append(weights.get(col, 0.0) * normalize(csr_matrix(M, dtype=float)))
    return hstack(blocks).tocsr(), vec

# Het blokgewicht bepaalt hoe zwaar elk blok meetelt in de cosine. Dat is iets anders dan de weging
# van bronnen later: hier gaat het om kolommen van X.
# De gewichten hieronder zijn een voorbeeld. Maak jullie eigen keuze en onderbouw die in het keuzelog.
WEIGHTS = {"summary": 1.0, "category": 0.3, "tags": 0.3}
X, VEC = build_vectors(books, WEIGHTS)

def similar_items(title, n=5):
    i = titles.index(title)
    sim = cosine_similarity(X[i], X).ravel()
    order = [j for j in np.argsort(-sim) if j != i][:n]
    return pd.DataFrame({"title": [titles[j] for j in order], "category": books.loc[order, "category"].values, "score": sim[order].round(2)})

print("X:", X.shape, "| blokken:", {col: len(v.vocabulary_) for col, v in VEC.items()})
similar_items("De draak onder de school")

## 1 · Een profiel van één titel

Een profiel is een rij met dezelfde kolommen als `X`. De simpelste: de rij van één titel. Cosine met alle titels, sorteren, klaar. Dat is precies `similar_items`, alleen begint het nu bij een *vector* in plaats van bij een *titel*.

In [11]:
def recommend(profile, n=10, exclude=()):
    """Top-n titles for a profile vector (one row, same columns as X)."""
    # exclude: posities van titels die niet in de lijst mogen. Hier komen later ook de harde filters terecht.
    sim = cosine_similarity(profile, X).ravel()
    order = [j for j in np.argsort(-sim) if j not in set(exclude)][:n]
    return pd.DataFrame({"title": [titles[j] for j in order],
                         "category": books.loc[order, "category"].values,
                         "score": sim[order].round(2)})

i = titles.index("De draak onder de school")
recommend(X[i], n=5, exclude=[i])          # zelfde lijst als similar_items

,title,category,score
0,De draak van de brandweer,humor,0.23
1,Draken bestaan niet,humor,0.16
2,De voetbalschool,realistisch,0.15
3,Code rood op de redactie,realistisch,0.12
4,Nieuw in de stad,realistisch,0.10


## 2 · Meer titels → één gemiddelde

Een paar rijen uit `X`, kolom voor kolom gemiddeld: één nieuwe rij. Dit is het principe achter elk profiel van vandaag: bij trefwoorden en genres kies je eerst de rijen, en neemt dan hun gemiddelde.

In [12]:
def build_profile(idx):
    """Mean of the rows of X at positions idx: one row, same columns as X."""
    return np.asarray(X[idx].mean(axis=0))

# Hier telt elke titel even zwaar. Je kunt titels ook wegen, bijvoorbeeld recente of hoog gewaardeerde
# titels zwaarder. Dat werken we hier niet uit.

drakenboeken = [titles.index(t) for t in ["Het drakenei", "De draak onder de school", "Draken bestaan niet", "De draak van de brandweer"]]
recommend(build_profile(drakenboeken), n=8, exclude=drakenboeken)

,title,category,score
0,Mijn broer de robot,humor,0.14
1,Code rood op de redactie,realistisch,0.11
2,De voetbalschool,realistisch,0.10
3,Het raadsel van de vuurtoren,spanning,0.09
4,De verdwenen schat van kapitein Bos,avontuur,0.07
5,De schoolreis naar de bergen,,0.07
6,Nieuw in de stad,realistisch,0.07
7,Het geheim van de oude molen,spanning,0.06


De gekozen titels zijn eruit gefilterd met `exclude`; anders stonden ze zelf bovenaan, want een titel lijkt sterk op een gemiddelde waar hij zelf in zit.

## 3 · Drie bronnen voor een profiel

Sam (2F, 13 jaar) kan op drie manieren zeggen wat hij wil. Alle drie worden een rij met dezelfde kolommen als `X`:

- **A · trefwoorden** → gemiddelde van alle titels die minstens één van die trefwoorden hebben
- **B · genres** → gemiddelde van alle titels met die genres: "het typische fantasy-boek"
- **C · eigen woorden** → dezelfde vectorizers als in `build_vectors`, op één tekst; het genre- en trefwoordblok blijven leeg

A en B werken hetzelfde: ze kiezen rijen via hun eigen blok (trefwoorden of genre) en nemen daarvan het gemiddelde. Het resultaat heeft in álle blokken waarden. C vult alleen het omschrijvingsblok.

Bij C zit de valkuil: je moet dezelfde vectorizers gebruiken (dezelfde kolommen!) en de blokken in dezelfde volgorde en met dezelfde blokgewichten aan elkaar plakken. Daarom geeft `build_vectors` nu `VEC` terug.

B (`profile_from_categories`) staat hieronder uitgewerkt als voorbeeld. A (`profile_from_tags`, in pseudocode) en C (`profile_from_text`, in stappen) schrijf je zelf. De cel onder de functies laat A, B en C naast elkaar zien, zodra je A en C hebt ingevuld; de cel na de opdracht is de test voor C.

In [15]:
def profile_from_tags(tags):
    """A: mean vector of all titles that have at least one of the given tags."""
    # wanted = de gevraagde trefwoorden, in kleine letters, als verzameling (set)
    # idx    = lege lijst
    # voor elk boek in books:
    #     als tag_list van dat boek minstens een trefwoord uit wanted bevat:
    #         voeg het rijnummer van dat boek toe aan idx
    # geef terug: build_profile(idx), idx
    raise NotImplementedError

def profile_from_categories(categories):
    """B: mean vector of all titles whose category is in the list."""
    idx = books.index[books["category"].isin([c.lower() for c in categories])].tolist()
    return build_profile(idx), idx

def profile_from_text(text, weights=WEIGHTS):
    """C: the same vectorizers as build_vectors, applied to one free text; genre and tag blocks stay empty."""
    # - Loop over de blokken in VEC, in dezelfde volgorde als build_vectors.
    # - Omschrijvingsblok: zet de tekst om met de vectorizer die al gefit is (transform, niet fit_transform).
    # - Genre- en trefwoordblok: maak een lege rij met dezelfde breedte als dat blok.
    # - Normaliseer elk blok en vermenigvuldig het met zijn gewicht, net als in build_vectors.
    # - Plak de blokken aan elkaar en geef een rij terug met dezelfde kolommen als X.
    raise NotImplementedError


In [ ]:
# A, B en C naast elkaar. Deze cel draait zodra je profile_from_tags en profile_from_text hebt ingevuld.
sam_A, idx_A = profile_from_tags(["draak", "school"])
sam_B, idx_B = profile_from_categories(["fantasy", "avontuur"])
sam_C = profile_from_text("iets spannends over draken, maar wel op een school en met vrienden erin")   # C geeft alleen een profiel terug, geen rijen
print("A kiest", len(idx_A), "titels, B kiest", len(idx_B), "titels")

pd.DataFrame({"A trefwoorden": recommend(sam_A, n=5)["title"].values,
              "B genres":      recommend(sam_B, n=5)["title"].values,
              "C eigen tekst": recommend(sam_C, n=5)["title"].values})


> **Opdracht 1 · profile_from_tags en profile_from_text.** Werk eerst `profile_from_tags` uit (de pseudocode hierboven); `profile_from_categories` is je voorbeeld. Bouw daarna `profile_from_text` voor jullie eigen `build_vectors`: dezelfde vectorizers, dezelfde blokvolgorde, dezelfde blokgewichten. Test: `profile_from_text(omschrijving van titel i)` moet titel i bovenaan zetten. Uitdaging: zoek een zoektekst waarbij C een titel bovenaan zet die A en B allebei missen.

In [ ]:
# test: de omschrijving van een titel als zoektekst moet die titel bovenaan zetten
for t in ["Leven in de ruimte", "Koken zonder recept", "De laatste wolf"]:
    profile_C = profile_from_text(books.loc[titles.index(t), "summary"])
    assert profile_C.shape == (1, X.shape[1]), ("zelfde kolommen als X", profile_C.shape, X.shape)
    top = recommend(profile_C, n=1)["title"][0]
    assert top == t, (t, top)
print("profile_from_text vindt de eigen omschrijving terug")

## 4 · De vragenlijst als dict, en één functie voor alles

Dit zijn de velden van de vragenlijst. Eén persoon is één dict. `person_profile` maakt een profiel per ingevuld veld en neemt daarvan het gemiddelde; `recommend_for` maakt er een lijst van.

`level` en `avoid` doen nog niets in het profiel: dat zijn harde filters. Die horen niet in `person_profile`, maar in `recommend_for`, vóór je de top n kiest (zie het commentaar in de code). De dummydata heeft geen kolom voor niveau of type materiaal, dus hier kun je ze niet uitproberen; in jullie eigen catalogus wel.

De bouwstenen heb je al: `profile_from_tags` (A), `profile_from_categories` (B) en `profile_from_text` (C). Wat nog ontbreekt is de functie die ze samenvoegt. Die schrijf je zelf; de cel met de vijf aanroepen eronder is je test.

In [ ]:
sam = {
    "name": "Sam",
    "tags": ["draak", "school", "vriendschap"],
    "genres": ["fantasy", "avontuur"],
    "free_text": "iets spannends over draken, maar wel op een school en met vrienden erin",
    "level": "2F",                  # harde filter: niet in het profiel, wel in recommend_for
    "avoid": ["tijdschrift"],       # harde filter: niet in het profiel, wel in recommend_for
}

def person_profile(person):
    """The profile vector for a person: the mean of the sources that are filled in."""
    # - Maak een profiel per ingevuld veld: tags (A), genres (B), free_text (C).
    # - Sla velden over die leeg zijn of ontbreken, en ook een bron die geen enkele titel vindt
    #   (een onbekend trefwoord of genre geeft een lege groep; daar bestaat geen gemiddelde van).
    # - Neem het gemiddelde van de profielen die er zijn: tel ze op en deel door het aantal.
    #   Dat kan omdat ze allemaal dezelfde kolommen hebben als X.
    # - Blijft er geen enkele bron over: geef een duidelijke foutmelding. recommend_for vangt die lezer op met cold_start.
    raise NotImplementedError

# WEGING: hierboven telt elke bron even zwaar. Dat is een keuze, en waarschijnlijk niet de beste.
# De weging werken we hier niet uit; die bouw je zelf (zie "Moet" onderaan). Denk na over:
#   - Welke bron zegt het meest over wat de lezer wil: trefwoorden, genres of eigen woorden? Waarom?
#   - Wat gebeurt er met de verhouding als een bron ontbreekt? (De gewichten van de bronnen die er wel
#     zijn moeten samen weer 1 zijn.)
#   - Hoe laat je zien dat jouw verhouding beter werkt dan gelijk wegen? Noteer het in het keuzelog.

def recommend_for(person, n=10):
    """Build a profile from whatever fields the person filled in, then recommend."""
    # - Is er niets ingevuld (of blijft er geen bron over): roep cold_start aan (deel 5) en geef dat terug.
    # - Bouw anders het profiel met person_profile.
    # - HARDE FILTERS: hier mag je titels verwijderen die nooit in de lijst mogen (bijvoorbeeld een te hoog
    #   level of een type uit avoid). Doe dat op de kandidaten, niet in person_profile: het profiel bouw je
    #   uit alle titels. Geef de posities die eruit moeten mee als exclude aan recommend.
    # - Geef de top n terug.
    raise NotImplementedError

> **Opdracht 2 · recommend_for.** Vul de vragenlijst voor jezelf in als dezelfde dict en zet hem in het gedeelde bestand van de klas (één rij per student; de kolomnamen zijn de keys). Schrijf jullie `recommend_for(person)` zo dat hij werkt met alleen `tags`, alleen `genres`, alleen `free_text`, en elke combinatie. Uitdaging: wat doe je als de bronnen elkaar tegenspreken (trefwoorden "koken" en "bakken", maar "fantasy" als genre)? Kies, en schrijf de reden op.

In [ ]:
# test: recommend_for moet werken voor elke combinatie van velden.
# De uitkomst kan anders zijn voor jullie eigen person_profile en recommend_for.
# De test is niet volledig, maar geeft een idee van wat er verwacht wordt.
for fields in [["tags"], ["genres"], ["free_text"], ["tags", "genres"], ["tags", "genres", "free_text"]]:
    person = {k: sam[k] for k in fields}
    result = list(recommend_for(person, n=3)["title"])
    assert len(result) == 3, fields
    print(fields, "→", result)

## 5 · Koude start: wat als er niets is ingevuld?

Een *koude start* is een lezer van wie je niets weet: geen trefwoorden, geen genres, geen eigen woorden. Het gemiddelde van niets bestaat niet, dus `person_profile` kan hier niet mee werken. Je hebt een andere route nodig. Twee aanpakken:

| Aanpak | Hoe werkt het | Wat heb je nodig | Zwak punt |
|---|---|---|---|
| **1 · Laat de lezer kiezen** | Toon een korte, gevarieerde lijst trefwoorden (of titels); de lezer kiest er 3. Daarmee heb je `tags` en werkt `person_profile` gewoon. | een extra stap in de app, en een keuzelijst die genoeg verschilt | de lezer moet iets doen voordat hij een lijst krijgt; haakt hij af, dan heb je niets |
| **2 · Vangnet: populair of gefilterd** | Geen profiel, maar een standaardlijst: populaire titels, eventueel gefilterd op bijvoorbeeld `level`. | een maat voor "populair" (uitleningen, beoordelingen) of een filter | iedereen krijgt (bijna) dezelfde lijst; zonder uitleendata is "populair" een gok |

Ze sluiten elkaar niet uit: eerst laten kiezen, en wie niets kiest krijgt het vangnet.

De vraag is dus niet alleen welke aanpak het beste aanbeveelt, maar ook **hoeveel lezers hij bereikt**. Neem dat mee in je keuze bij opdracht 3.

In [ ]:
def cold_start(person, n=5):
    """Aanbevelingen voor een lezer die niets invulde.

    Geef een DataFrame terug met in elk geval de kolom "title", net als recommend; daar rekenen de tests op.

    Vul zelf aan. Je kunt hergebruiken wat hierboven staat:
      - recommend(profile, n=n)                 -> DataFrame met de top-n titels
      - person_profile({"tags": [...]})  -> profiel uit gekozen trefwoorden (aanpak 1)
      - books, titles                           -> de catalogus (aanpak 2)
    """
    # Aanpak 1: laat de lezer kiezen
    #   - maak een gevarieerde keuzelijst, bijvoorbeeld de vaakst voorkomende trefwoorden per genre
    #   - in een app kiest de lezer er 3; hier kun je ze zelf kiezen om te testen
    #   - roep person_profile aan met {"tags": keuze} en geef het profiel aan recommend

    # Aanpak 2: vangnet
    #   - geen profiel; geef een standaardlijst
    #   - hoe bepaal je "populair" als je geen uitleendata hebt? Noteer je keuze in het keuzelog

    raise NotImplementedError

> **Opdracht 3 · kies een koude start.** Kies aanpak 1, 2 of beide of nog iets anders en bouw hem in `recommend_for`.

In [ ]:
# test: ook een lezer die niets invulde krijgt een lijst
for person in [{}, {"tags": [], "genres": [], "free_text": ""}]:
    result = list(recommend_for(person, n=5)["title"])
    assert len(result) == 5, person
    assert len(set(result)) == 5, ("geen dubbele titels", person)
    print(person, "→", result)

---
## Opdracht

**Doel.** Jullie systeem geeft een lijst voor een persoon uit trefwoorden, genres of eigen woorden, ook voor iemand die niets invulde.

**Moet** (in `recommender.py`)
- [ ] `build_profile(idx)`: gemiddelde van de gekozen rijen van `X`.
- [ ] `profile_from_tags(tags)` en `profile_from_categories(categories)`: profiel uit trefwoorden en uit genres.
- [ ] `profile_from_text(text)`: een profiel uit een eigen tekst, met dezelfde vectorizers en blokvolgorde als `build_vectors`.
- [ ] `person_profile(person)`: één profiel uit de bronnen die zijn ingevuld.
- [ ] `recommend_for(person)`: werkt voor elke combinatie van `tags`, `genres` en `free_text`.
- [ ] `cold_start(person)`: een lijst voor een lezer die niets invulde; `recommend_for` roept hem aan.
- [ ] Weging: laat niet elke bron even zwaar tellen. Dit staat niet uitgewerkt in het notebook: bedenk zelf de verhouding (zie de vragen bij `person_profile`), bouw hem in, en onderbouw hem in het keuzelog.

**Kan**
- [ ] Een profiel dat uit twee delen bestaat (twee gemiddelden, twee top 5's samengevoegd) voor een lezer met twee smaken.

*Klaar als.* `recommend_for` werkt voor elke combinatie van velden en voor een lege gebruiker.

In [ ]:
# Startpunt voor recommender.py. Jullie kolommen heten anders; haal de namen uit jullie rollentabel.

def build_profile(idx):
    """Mean of the rows of X at positions idx: one row, same columns as X."""
    # - Pak de rijen van X voor de gekozen titels.
    # - Neem het gemiddelde van die rijen.
    # - Geef een rij terug met dezelfde kolommen als X.
    raise NotImplementedError

def profile_from_tags(tags):
    """A: mean vector of all titles that have at least one of the given tags."""
    # - Zoek de titels die minstens één van de trefwoorden hebben.
    # - Geef build_profile van die rijen terug (en de gekozen rijen).
    raise NotImplementedError

def profile_from_categories(categories):
    """B: mean vector of all titles whose genre is in the list."""
    # - Zoek de titels met een van die genres.
    # - Geef build_profile van die rijen terug (en de gekozen rijen).
    raise NotImplementedError

def profile_from_text(text):
    """C: same vectorizers and block order as build_vectors, applied to one free text."""
    # - Omschrijvingsblok: transform met de vectorizer uit build_vectors (VEC).
    # - Genre- en trefwoordblok: leeg, maar wel even breed.
    # - Normaliseer elk blok en vermenigvuldig het met zijn gewicht.
    # - Plak de blokken in dezelfde volgorde aan elkaar.
    raise NotImplementedError

def person_profile(person):
    """The profile vector for a person: a (weighted) mix of the sources that are filled in."""
    # - Maak een profiel per ingevuld veld: tags, genres en free_text.
    # - Sla lege velden over, en bronnen die geen enkele titel vinden.
    # - Voeg de profielen samen met jullie eigen weging; de gewichten van de bronnen die er zijn tellen op tot 1.
    # - Blijft er geen bron over: geef een foutmelding.
    raise NotImplementedError

def recommend_for(person, n=10):
    """person: dict with the questionnaire fields. Works for any combination of filled fields; cold start when none."""
    # - Is er niets ingevuld (of blijft er geen bron over), roep dan cold_start aan.
    # - Bouw anders het profiel met person_profile.
    # - HARDE FILTERS: hier mag je titels verwijderen die nooit in de lijst mogen (bijvoorbeeld een te hoog
    #   level of een type uit avoid), vóór je de top n kiest. Niet in het profiel: dat bouw je uit alle titels.
    # - Geef de top n terug.
    raise NotImplementedError

def cold_start(person, n=5):
    """Lijst voor een lezer die niets invulde. Jouw gekozen aanpak (1, 2 of beide)."""
    # - Aanpak 1: toon een gevarieerde keuzelijst trefwoorden, laat er 3 kiezen en bouw daar een profiel van.
    # - Aanpak 2: geef een standaardlijst terug, zonder profiel.
    # - Aanpak 3: anders, jullie eigen keuze
    # - Geef de top n terug als DataFrame met de kolom "title", net als recommend.
    raise NotImplementedError